# 7일 → 7일 종가 분류 모델 (dense + 마스킹 RMSNorm)

`ml/train.csv · eval.csv · test.csv` 를 읽어 7일 입력 → 7일 출력(다음날 종가 분류)을 학습한다.

## 설계 요약
- Date 제외 컬럼(=종목) 순회. 각 종목에서 슬라이딩 창(간격 1) 생성.
- 1개 샘플 = 연속 8개 달력행(t0~t7). 입력 t0~t6, 출력 t1~t7.
  - 8행의 Date 가 모두 +1일 연속일 때만 유효(분기 셔플 경계 점프 제외).
  - 값은 round 후 int. 결측/휴장/상장전은 -1 그대로.
  - 입력 7개가 모두 -1 이면 그 창은 제외.
- 전 종목 창을 하나로 pooling 하여 단일 dense 모델 학습.
- 출력층 softmax 클래스 = train 출력에 등장한 '원본 정수값' vocabulary.
- 입력 정규화 = 마스킹 RMSNorm. 입력의 -1 은 마스킹(반영 안 함, dropout 식).
- 손실 = 타임스텝별 CE. 출력값 -1(및 vocab 밖)은 연산 제외.
  겹치는 구간(t1~t6)은 가중치↓, 마지막 예측(t7)은 가중치↑.
- 지표 = top-1/3/5 정확도(전체·마지막스텝). eval 의 '마지막스텝(t7)' top-3·top-5 가 직전 best 대비 둘 다 오를 때만 best 갱신.

> 실행 환경: torch 가 설치된 `.conda` 환경의 python 으로 실행. GPU 가 있으면 자동 사용(`DEVICE`).

In [ ]:
import shutil
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

## Config

In [ ]:
# 로컬 산출물(학습 중 best 체크포인트)용 경로
HERE       = os.getcwd()
BEST_DIR   = os.path.join(HERE, 'best')
BEST_CKPT  = os.path.join(BEST_DIR, 'best_model.pt')
BEST_META  = os.path.join(BEST_DIR, 'best_meta.json')

# ── 데이터: git 현재 브랜치에서 ml/*.csv 3개'만' sparse-checkout 으로 가져오기 ──
#   * 저장소가 private 이면 REPO_URL 에 토큰 필요:
#     https://<TOKEN>@github.com/syuka-fan/stock.git
#   * ml/*.csv 가 해당 브랜치에 커밋/푸시되어 있어야 함.
import subprocess
REPO_URL  = 'https://github.com/syuka-fan/stock.git'   # origin
BRANCH    = 'main'                                      # 현재 브랜치
DATA_REPO = os.path.join(HERE, 'stock_repo')            # 체크아웃 위치
FILES     = ['ml/train.csv', 'ml/eval.csv', 'ml/test.csv']

def fetch_data_from_git():
    if not os.path.isdir(os.path.join(DATA_REPO, '.git')):
        # blob 필터 + sparse + shallow: 히스토리/불필요 파일 없이 최소 클론
        subprocess.run(['git', 'clone', '--filter=blob:none', '--sparse',
                        '--depth', '1', '-b', BRANCH, REPO_URL, DATA_REPO], check=True)
    # 지정한 파일들'만' 작업트리에 materialize
    subprocess.run(['git', '-C', DATA_REPO, 'sparse-checkout', 'set',
                    '--no-cone', *FILES], check=True)
    return {os.path.basename(f): os.path.join(DATA_REPO, f) for f in FILES}

_paths    = fetch_data_from_git()
TRAIN_CSV = _paths['train.csv']
EVAL_CSV  = _paths['eval.csv']
TEST_CSV  = _paths['test.csv']
print('데이터 경로:', TRAIN_CSV, EVAL_CSV, TEST_CSV, sep='\n  ')

IN_LEN     = 7          # 입력 길이 (t0~t6)
OUT_LEN    = 7          # 출력 길이 (t1~t7)
STRIDE     = 1          # 슬라이딩 간격
MISSING    = -1         # 결측 sentinel

W_OVERLAP  = 0.5        # 겹치는 구간(t1~t6) 손실 가중치 (낮게)
W_LAST     = 2.0        # 마지막 예측(t7) 손실 가중치 (높게)

HIDDEN     = [256, 256] # dense hidden 차원
DROP_P     = 0.1        # dense dropout
MASK_RESCALE = True     # 마스킹 시 dropout 식 스케일 보정(IN_LEN/유효개수)

EPOCHS     = 40
BATCH      = 512
LR         = 1e-3
WEIGHT_DECAY = 1e-5
SEED       = 42
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IGNORE     = -100       # 손실/평가에서 제외할 타깃 인덱스
EPS        = 1e-6

torch.manual_seed(SEED); np.random.seed(SEED)
print('DEVICE =', DEVICE)

## 창(window) 생성
Date 제외 컬럼을 순회하며, 연속 8개 달력행(모두 +1일)에서만 창을 만든다.

In [ ]:
def build_windows(csv_path):
    """CSV → (X, Y). X,Y: (N, IN_LEN)/(N, OUT_LEN) int64, -1 은 결측 유지."""
    df = pd.read_csv(csv_path, parse_dates=['Date']).sort_values('Date')
    dates = df['Date'].values.astype('datetime64[D]')
    # 연속 행 간 날짜 차이(일). day_gap[i] = date[i+1] - date[i]
    day_gap = (dates[1:] - dates[:-1]).astype('timedelta64[D]').astype(np.int64)

    cols = [c for c in df.columns if c != 'Date']
    span = IN_LEN + 1                 # 한 샘플이 차지하는 연속 행 수 (t0~t7 = 8)

    X_list, Y_list = [], []
    for c in cols:
        # round 후 정수화(코스피 지수 등 실수 → 반올림). 결측 -1 은 그대로 int.
        vals = np.rint(pd.to_numeric(df[c], errors='coerce').values).astype(np.int64)
        n = len(vals)
        for i in range(0, n - span + 1, STRIDE):
            # t0..t7 (i .. i+span-1) 이 모두 +1일 연속인지 확인
            if not np.all(day_gap[i:i + span - 1] == 1):
                continue
            x = vals[i:i + IN_LEN]              # t0~t6
            y = vals[i + 1:i + 1 + OUT_LEN]     # t1~t7
            if np.all(x == MISSING):           # 입력 전부 결측이면 제외
                continue
            X_list.append(x)
            Y_list.append(y)

    if not X_list:
        return np.empty((0, IN_LEN), np.int64), np.empty((0, OUT_LEN), np.int64)
    return np.stack(X_list), np.stack(Y_list)

## Vocabulary (원본 정수값 → 클래스 인덱스)
train 출력의 고유 정수값(오름차순)을 클래스로. 클래스 인덱스 = `vocab_vals` 내 위치.
-1 및 vocab 밖 값은 IGNORE 로 매핑되어 손실/평가에서 제외된다.

In [ ]:
def build_vocab(Y_train):
    """train 출력의 고유 정수값(결측 -1 제외, 오름차순) → 클래스."""
    return np.unique(Y_train[Y_train != MISSING]).astype(np.int64)  # sorted unique


def map_targets(Y, vocab_vals):
    """출력 정수 → 클래스 인덱스. -1/ vocab 밖 → IGNORE. (searchsorted 벡터화)"""
    out = np.full(Y.shape, IGNORE, dtype=np.int64)
    flat = Y.ravel()
    pos = np.searchsorted(vocab_vals, flat)
    pos_c = np.clip(pos, 0, len(vocab_vals) - 1)
    match = (vocab_vals[pos_c] == flat) & (flat != MISSING)
    out_flat = out.ravel()
    out_flat[match] = pos_c[match]
    return out_flat.reshape(Y.shape)


def to_tensors(X, Yidx):
    Xf = torch.from_numpy(X.astype(np.float32))     # 원본 정수(실수형), -1 sentinel 포함
    mask = (Xf != MISSING).float()                  # 1=유효, 0=결측(-1)
    Yt = torch.from_numpy(Yidx)                     # (N, OUT_LEN) 클래스 인덱스/IGNORE
    return Xf, mask, Yt

## 모델: 마스킹 RMSNorm → dense MLP → (OUT_LEN × vocab)

In [ ]:
class RMSNorm(nn.Module):
    """표준 Root Mean Square Normalization (학습 가능 gain)."""
    def __init__(self, dim, eps=EPS):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


def masked_rms_norm(x, mask, rescale=MASK_RESCALE, eps=EPS):
    """입력 RMSNorm + 마스킹. -1(결측) 위치는 RMS 계산에서 뺀 뒤 0 으로.

    dropout 처럼, 남은(유효) 위치를 IN_LEN/유효개수 로 스케일 보정(옵션).
    """
    valid = mask                                   # (B, L) 1/0
    cnt = valid.sum(dim=-1, keepdim=True).clamp_min(1.0)
    xz = x * valid                                 # 결측 → 0 (제곱합에서 제외)
    ms = (xz.pow(2).sum(dim=-1, keepdim=True)) / cnt
    rms = torch.sqrt(ms + eps)
    xn = (xz / rms) * valid                        # 정규화 후 결측 위치 다시 0
    if rescale:
        xn = xn * (x.shape[-1] / cnt)              # inverted-dropout 식 보정
    return xn


class DenseForecaster(nn.Module):
    """마스킹 RMSNorm → MLP → (OUT_LEN × vocab) 로짓."""
    def __init__(self, in_len, out_len, vocab, hidden, drop_p):
        super().__init__()
        self.out_len, self.vocab = out_len, vocab
        layers, d = [], in_len
        for h in hidden:
            layers += [nn.Linear(d, h), RMSNorm(h), nn.GELU(), nn.Dropout(drop_p)]
            d = h
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Linear(d, out_len * vocab)

    def forward(self, x, mask):
        h = masked_rms_norm(x, mask)               # 입력 정규화 + 마스킹
        h = self.backbone(h)
        logits = self.head(h)                      # (B, out_len*vocab)
        return logits.view(-1, self.out_len, self.vocab)

## 손실 / 지표
겹치는 구간(t1~t6) 가중치↓, 마지막(t7) 가중치↑. 타깃 -1/OOV 는 제외.

In [ ]:
def position_weights(device):
    """겹치는 구간(t1~t6) 낮게, 마지막(t7) 높게."""
    w = torch.full((OUT_LEN,), W_OVERLAP, device=device)
    w[-1] = W_LAST
    return w


def weighted_masked_loss(logits, targets, pos_w):
    """타임스텝별 CE에 위치가중치 + 타깃 유효(≠IGNORE) 마스킹."""
    B, L, V = logits.shape
    ce = F.cross_entropy(
        logits.reshape(B * L, V), targets.reshape(B * L),
        ignore_index=IGNORE, reduction='none').view(B, L)      # (B, L)
    valid = (targets != IGNORE).float()
    w = pos_w.unsqueeze(0) * valid                             # (B, L)
    denom = w.sum().clamp_min(1.0)
    return (ce * w).sum() / denom


@torch.no_grad()
def topk_stats(logits, targets, ks=(1, 3, 5)):
    """전체 유효 위치 및 마지막스텝(t7) top-k 정답 카운트."""
    B, L, V = logits.shape
    maxk = max(ks)
    topk = logits.topk(maxk, dim=-1).indices                  # (B, L, maxk)
    valid = targets != IGNORE                                 # (B, L)
    hit = (topk == targets.unsqueeze(-1))                     # (B, L, maxk)
    res = {}
    last = torch.zeros(L, dtype=torch.bool); last[-1] = True
    for name, sel in (('all', torch.ones(L, dtype=torch.bool)), ('last', last)):
        m = valid & sel.to(valid.device)
        res[name] = {'total': int(m.sum())}
        for k in ks:
            res[name][k] = int((hit[..., :k].any(-1) & m).sum())
    return res


def eval_loader(model, loader, pos_w):
    model.eval()
    agg = {'all': {'total': 0, 1: 0, 3: 0, 5: 0},
           'last': {'total': 0, 1: 0, 3: 0, 5: 0}}
    loss_sum, loss_n = 0.0, 0
    with torch.no_grad():
        for xb, mb, yb in loader:
            xb, mb, yb = xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb, mb)
            loss_sum += float(weighted_masked_loss(logits, yb, pos_w)) * xb.size(0)
            loss_n += xb.size(0)
            s = topk_stats(logits, yb)
            for grp in ('all', 'last'):
                for key in agg[grp]:
                    agg[grp][key] += s[grp][key]
    acc = {}
    for grp in ('all', 'last'):
        t = max(agg[grp]['total'], 1)
        acc[grp] = {k: agg[grp][k] / t for k in (1, 3, 5)}
        acc[grp]['total'] = agg[grp]['total']
    return loss_sum / max(loss_n, 1), acc


def fmt(acc):
    # 라벨 top1/top3/top5 = top-1/3/5 정확도(타임스텝 t1~t7 과 혼동 방지). all=전체, last=마지막스텝(t7)
    a, l = acc['all'], acc['last']
    return (f"all[top1:{a[1]:.4f} top3:{a[3]:.4f} top5:{a[5]:.4f}] "
            f"last[top1:{l[1]:.4f} top3:{l[3]:.4f} top5:{l[5]:.4f}]")

## 데이터 로드 · 창 생성 · vocab · 텐서

In [ ]:
print('창 생성 중...')
Xtr, Ytr = build_windows(TRAIN_CSV)
Xev, Yev = build_windows(EVAL_CSV)
Xte, Yte = build_windows(TEST_CSV)
print(f'  train {Xtr.shape[0]}  eval {Xev.shape[0]}  test {Xte.shape[0]} 창')

vocab_vals = build_vocab(Ytr)
V = len(vocab_vals)
print(f'  vocab(원본 정수값 클래스 수) = {V}')

tr = to_tensors(Xtr, map_targets(Ytr, vocab_vals))
ev = to_tensors(Xev, map_targets(Yev, vocab_vals))
te = to_tensors(Xte, map_targets(Yte, vocab_vals))

tr_ld = DataLoader(TensorDataset(*tr), batch_size=BATCH, shuffle=True)
ev_ld = DataLoader(TensorDataset(*ev), batch_size=BATCH)
te_ld = DataLoader(TensorDataset(*te), batch_size=BATCH)

## 모델 · 옵티마이저

In [ ]:
os.makedirs(BEST_DIR, exist_ok=True)
model = DenseForecaster(IN_LEN, OUT_LEN, V, HIDDEN, DROP_P).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
pos_w = position_weights(DEVICE)
print(f'파라미터 수 = {sum(p.numel() for p in model.parameters()):,}')

## 학습 루프
eval 마지막스텝(t7) top-3 · top-5 가 둘 다 직전 best 보다 오를 때만 best 갱신.

In [ ]:
best_top3, best_top5 = -1.0, -1.0  # eval 마지막스텝(t7) 기준
for ep in range(1, EPOCHS + 1):
    model.train()
    run, nb = 0.0, 0
    for xb, mb, yb in tr_ld:
        xb, mb, yb = xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = weighted_masked_loss(model(xb, mb), yb, pos_w)
        loss.backward(); opt.step()
        run += float(loss); nb += 1
    ev_loss, ev_acc = eval_loader(model, ev_ld, pos_w)

    # best 갱신: eval 마지막스텝 top-3·top-5 가 둘 다 상승할 때만
    top3, top5 = ev_acc['last'][3], ev_acc['last'][5]
    improved = (top3 > best_top3) and (top5 > best_top5)
    tag = ''
    if improved:
        best_top3, best_top5 = top3, top5
        torch.save({'model': model.state_dict(),
                    'vocab_vals': vocab_vals,
                    'config': {'IN_LEN': IN_LEN, 'OUT_LEN': OUT_LEN,
                               'HIDDEN': HIDDEN, 'vocab': V}}, BEST_CKPT)
        with open(BEST_META, 'w', encoding='utf-8') as f:
            json.dump({'epoch': ep, 'eval_last_top3': top3, 'eval_last_top5': top5,
                       'eval_acc': ev_acc}, f, ensure_ascii=False, indent=2)
        tag = '  <== best 갱신'
    print(f'ep {ep:3d} | train_loss {run/max(nb,1):.4f} | '
          f'eval_loss {ev_loss:.4f} | {fmt(ev_acc)}{tag}')

## best 로 test 평가

In [ ]:
if os.path.exists(BEST_CKPT):
    ck = torch.load(BEST_CKPT, weights_only=False)
    model.load_state_dict(ck['model'])
te_loss, te_acc = eval_loader(model, te_ld, pos_w)
print(f'[TEST] loss {te_loss:.4f} | {fmt(te_acc)}')

## test 마지막스텝(t7) 예측의 R²
best 모델로 test 를 추론해, 마지막 예측(t7)의 argmax 클래스를 원본 정수 예측값으로 환산하고
원본 정수 실제값과 R² 를 계산한다. **원본이 -1인 결측치는 제외.**

In [ ]:
# test 마지막스텝(t7) 예측의 R^2 (원본 -1 결측 제외)
#   예측값 = 모델이 고른 클래스(argmax=top-1)의 원본 정수값
#   실제값 = 원본 정수 종가 t7 (Yte[:, -1], 클래스 인덱스가 아니라 '원본 값')
model.eval()
pred_idx = []
with torch.no_grad():
    Xf_te, mask_te = te[0], te[1]              # to_tensors 순서 = Xte/Yte 행 순서 유지
    for i in range(0, Xf_te.size(0), BATCH):
        logits = model(Xf_te[i:i + BATCH].to(DEVICE), mask_te[i:i + BATCH].to(DEVICE))
        pred_idx.append(logits[:, -1, :].argmax(-1).cpu().numpy())   # 마지막스텝 argmax
pred_idx = np.concatenate(pred_idx)

y_pred = vocab_vals[pred_idx].astype(np.float64)   # 클래스 인덱스 → 원본 정수 예측값
y_true = Yte[:, -1].astype(np.float64)             # 원본 정수 실제값(t7)

m = y_true != MISSING                              # 원본 -1 결측 제외
yt, yp = y_true[m], y_pred[m]
ss_res = np.sum((yt - yp) ** 2)
ss_tot = np.sum((yt - yt.mean()) ** 2)
r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
print(f'[TEST] 마지막스텝(t7) R^2 = {r2:.6f}  (n={int(m.sum())}, 원본 -1 제외)')

## 학습된 모델을 Google Drive 에 저장
(상단에서 마운트한) Drive 에 best 체크포인트(+최종 상태)를 저장하고, `flush_and_unmount()` 로 실제 반영한다.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/stock_model'   # 저장 위치(원하면 변경)
os.makedirs(DRIVE_DIR, exist_ok=True)

# 학습 중 갱신된 best 체크포인트/메타 복사
if os.path.exists(BEST_CKPT):
    shutil.copy(BEST_CKPT, os.path.join(DRIVE_DIR, 'best_model.pt'))
if os.path.exists(BEST_META):
    shutil.copy(BEST_META, os.path.join(DRIVE_DIR, 'best_meta.json'))

# 현재 메모리의 모델 상태도 최종본으로 저장(vocab·config 포함)
torch.save({'model': model.state_dict(),
            'vocab_vals': vocab_vals,
            'config': {'IN_LEN': IN_LEN, 'OUT_LEN': OUT_LEN,
                       'HIDDEN': HIDDEN, 'vocab': V}},
           os.path.join(DRIVE_DIR, 'model_pretrain.pt'))
print('Drive 저장 완료 →', DRIVE_DIR)

# 버퍼 flush 후 언마운트(실제 디스크 반영)
drive.flush_and_unmount()
print('flush_and_unmount 완료')